## Upgrade your model to the latest Wflow.jl version

HydroMT-Wflow provides a dedicated CLI command to upgrade your Wflow.jl model to the latest version: `hydromt_wflow upgrade`.

For SBM models, the main difference between Wflow version 0.8.1 and earlier versus version 1.0.0 and later is that the TOML file structure was updated with new sections and standard names for internal variable references. For example *lateral.river.q_av* is now *river_water__volume_flow_rate*.

For sediment models, apart from the TOML changes, estimation of some parameters has been moved outside of the Wflow.jl code and into HydroMT-Wflow, allowing easier adjustment and calibration.

The upgrade command copies your model to a new output directory and applies all necessary upgrade steps automatically. Let's see how to use it.

### Upgrading a Wflow SBM model

For a detailed overview of the upgrade cli options, run:

```sh
hydromt_wflow upgrade --help
```

For a basic SBM upgrade, no configuration file is needed. The upgrade command detects the model version and applies all necessary steps:

```sh
hydromt_wflow upgrade "./data/wflow_upgrade/sbm" -o "./data/wflow_upgrade/sbm_v1" -v
```

Let's first look at the original TOML file before upgrading:

In [ ]:
from pathlib import Path
import tempfile

v0_upgrade_dir = Path("./data/wflow_upgrade")
v1_upgrade_dir = Path(tempfile.mkdtemp(prefix="wflow_upgrade_"))

toml_v0x = v0_upgrade_dir / "sbm" / "wflow_sbm.toml"
print(toml_v0x.read_text())

Now let's run the upgrade. If the config filename differs from the default (*wflow_sbm.toml*), you can specify it via a config file passed with `-i`. Here we use `wflow_sbm_v0x.toml` as the config filename:

In [ ]:
from hydromt_wflow import cli
cli.upgrade(
    args=[
        str(v0_upgrade_dir / "sbm"),
        "-o", str(v1_upgrade_dir / "sbm"),
        "-v",
    ],
    standalone_mode=False
)

And let's see the results. Here is what our old TOML file looked like:

In [ ]:
toml_v0x = v0_upgrade_dir / "sbm" / "wflow_sbm.toml"
print(toml_v0x.read_text())

And here is the same TOML in version 1 format:

In [ ]:
toml_v1 = v1_upgrade_dir / "sbm" / "wflow_sbm.toml"
print(toml_v1.read_text())

The lakes and reservoirs have been merged into one structure and the new staticmaps file has been generated. Here are all the available variables:

In [ ]:
import xarray as xr

staticmaps = xr.open_dataset(v1_upgrade_dir / "sbm" / "staticmaps.nc")
print(list(staticmaps.data_vars.keys()))

### Upgrading a Wflow Sediment model

For sediment models, you may need to pass upgrade options (e.g. which soil dataset to use). Create a simplified YAML config file:

In [ ]:
fn_config = v0_upgrade_dir / "wflow_upgrade_sediment.yml"
print(fn_config.read_text())

This config sets version-specific options for the upgrade. On the command line:

```sh
hydromt_wflow upgrade "./data/wflow_upgrade/sediment" -o "./data/wflow_upgrade/sediment_v1" --model-type wflow_sediment -i wflow_upgrade_sediment.yml -d artifact_data -v
```

Let's run the upgrade:

In [ ]:
cli.upgrade(args=[
    str(v0_upgrade_dir / "sediment"),
    "-o", str(v1_upgrade_dir / "sediment"),
    "--model-type", "wflow_sediment",
    "-i", str(fn_config),
    "-d", "artifact_data",
    "-v",
], standalone_mode=False)

And let's see the results. Here is what our old TOML file looked like:

In [ ]:
toml_v0x = v0_upgrade_dir / "sediment" / "wflow_sediment.toml"
print(toml_v0x.read_text())

And here is the same TOML in version 1 format:

In [ ]:
toml_v1 = v1_upgrade_dir / "sediment" / "wflow_sediment.toml"
print(toml_v1.read_text())

You can also note that the new staticmaps file was generated. Here are all the available variables:

In [ ]:
import xarray as xr

staticmaps = xr.open_dataset(v1_upgrade_dir / "sediment" / "staticmaps.nc")
print(list(staticmaps.data_vars.keys()))